In [1]:
import os
import pandas as pd
from collections import defaultdict, Counter
import random

# ------------------------------
# Pairwise Comparison Generator
# ------------------------------
def generate_pairwise_comparisons_from_df(
    df: pd.DataFrame,
    id_col: str = "individual_id",
    group_sizes=[3, 5, 7, 10],
    n_repeats=5,
    mode='both',
    selfmatch_factor=2.0
):
    individuals = defaultdict(list)
    for idx, row in df.iterrows():
        individuals[row[id_col]].append(idx)

    comparisons = []

    for size_a in group_sizes:
        for size_b in group_sizes:
            if mode == 'symmetric' and size_a != size_b:
                continue
            if mode == 'asymmetric' and size_a == size_b:
                continue

            eligible_inds_a = [ind for ind, samples in individuals.items() if len(samples) >= size_a]
            eligible_inds_b = [ind for ind, samples in individuals.items() if len(samples) >= size_b]

            for ind_a in eligible_inds_a:
                for ind_b in eligible_inds_b:
                    if ind_a >= ind_b:
                        continue
                    for _ in range(n_repeats):
                        samples_a = random.sample(individuals[ind_a], size_a)
                        samples_b = random.sample(individuals[ind_b], size_b)
                        comparisons.append({
                            'ind_a': ind_a,
                            'ind_b': ind_b,
                            'size_a': size_a,
                            'size_b': size_b,
                            'samples_a': samples_a,
                            'samples_b': samples_b,
                            'same_individual': False
                        })

            for ind in individuals:
                if len(individuals[ind]) < size_a + size_b:
                    continue
                self_repeats = int(n_repeats * selfmatch_factor)
                for _ in range(self_repeats):
                    combined = random.sample(individuals[ind], size_a + size_b)
                    samples_a = combined[:size_a]
                    samples_b = combined[size_a:]
                    comparisons.append({
                        'ind_a': ind,
                        'ind_b': ind,
                        'size_a': size_a,
                        'size_b': size_b,
                        'samples_a': samples_a,
                        'samples_b': samples_b,
                        'same_individual': True
                    })

    return comparisons


In [2]:
# train.csv einlesen
df = pd.read_csv(r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\train.csv")

# Pairwise-Vergleiche erzeugen
comparisons = generate_pairwise_comparisons_from_df(
    df,
    group_sizes=[3, 5, 7, 10],
    n_repeats=2,
    mode="both",
    selfmatch_factor=10
)

# Beispielausgabe
print(f"Erzeugte Vergleiche: {len(comparisons)}")
print("Beispielvergleich:", comparisons[0])

# Vergleiche speichern
comparisons_df = pd.DataFrame(comparisons)
save_path = r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\pairwise\pairwise_comparisons_otter.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
comparisons_df.to_csv(save_path, index=False)
print(f"Vergleiche gespeichert nach: {save_path}")

# ------------------------------
# Sample Usage Auswertung
# ------------------------------
# Alle IDs aus Vergleichen extrahieren
all_ids = []
for comp in comparisons:
    all_ids.extend([comp['ind_a']] * comp['size_a'])
    all_ids.extend([comp['ind_b']] * comp['size_b'])

# Zähle Nutzung pro Individuum
sample_use_counts = Counter(all_ids)
sample_use_df = pd.DataFrame.from_dict(sample_use_counts, orient='index', columns=['sample_usages'])
sample_use_df.index.name = 'individual_id'
sample_use_df = sample_use_df.sort_values('sample_usages', ascending=False)

# Ausgabe der Top 10
print("\nTop 10 Individuen nach Sample-Nutzung:")
print(sample_use_df.head(10).to_string())

# Optional speichern
usage_path = save_path.replace("pairwise_comparisons_otter.csv", "pairwise_sample_usage.csv")
sample_use_df.to_csv(usage_path)

Erzeugte Vergleiche: 12876
Beispielvergleich: {'ind_a': 'zanetka', 'ind_b': 'zewe', 'size_a': 3, 'size_b': 3, 'samples_a': [3, 0, 1], 'samples_b': [283, 274, 287], 'same_individual': False}
Vergleiche gespeichert nach: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\pairwise\pairwise_comparisons_otter.csv

Top 10 Individuen nach Sample-Nutzung:
               sample_usages
individual_id               
sam                     8450
rosie                   8450
otella                  8450
innsbruck_1             8450
japet                   8050
paulinchen              8050
zewe                    8050
shorfy                  7370
anton                   7370
mette                   7370


In [4]:
# train.csv einlesen
df = pd.read_csv(r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\train.csv")

# Pairwise-Vergleiche erzeugen
comparisons = generate_pairwise_comparisons_from_df(
    df,
    group_sizes=[3, 5, 7, 10],
    n_repeats=3,
    mode="both",
    selfmatch_factor=20
)

# Beispielausgabe
print(f"Erzeugte Vergleiche: {len(comparisons)}")
print("Beispielvergleich:", comparisons[0])

# Vergleiche speichern
comparisons_df = pd.DataFrame(comparisons)
save_path = r"C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\pairwise\pairwise_comparisons_otter.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
comparisons_df.to_csv(save_path, index=False)
print(f"Vergleiche gespeichert nach: {save_path}")

# ------------------------------
# Sample Usage Auswertung
# ------------------------------
# Alle IDs aus Vergleichen extrahieren
all_ids = []
for comp in comparisons:
    all_ids.extend([comp['ind_a']] * comp['size_a'])
    all_ids.extend([comp['ind_b']] * comp['size_b'])

# Zähle Nutzung pro Individuum
sample_use_counts = Counter(all_ids)
sample_use_df = pd.DataFrame.from_dict(sample_use_counts, orient='index', columns=['sample_usages'])
sample_use_df.index.name = 'individual_id'
sample_use_df = sample_use_df.sort_values('sample_usages', ascending=False)

# Ausgabe der Top 10
print("\nTop 10 Individuen nach Sample-Nutzung:")
print(sample_use_df.head(10).to_string())

# Optional speichern
usage_path = save_path.replace("pairwise_comparisons_otter.csv", "pairwise_sample_usage.csv")
sample_use_df.to_csv(usage_path)
print(f"Sample-Nutzung gespeichert nach: {usage_path}")

Erzeugte Vergleiche: 26154
Beispielvergleich: {'ind_a': 'zanetka', 'ind_b': 'zewe', 'size_a': 3, 'size_b': 3, 'samples_a': [3, 1, 5], 'samples_b': [271, 283, 285], 'same_individual': False}
Vergleiche gespeichert nach: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\pairwise\pairwise_comparisons_otter.csv

Top 10 Individuen nach Sample-Nutzung:
               sample_usages
individual_id               
sam                    18675
rosie                  18675
otella                 18675
innsbruck_1            18675
japet                  17475
paulinchen             17475
zewe                   17475
shorfy                 15435
anton                  15435
mette                  15435
Sample-Nutzung gespeichert nach: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\splits\eurasian_otter_(1)\pairwise\pairwise_sample_usage.csv


SyntaxError: incomplete input (616974813.py, line 107)